In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [2]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
from siuba import *

fs = get_fs()

In [3]:
import altair as alt
import folium

In [4]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [5]:
import _replica_utils

In [6]:
from IPython.display import HTML

In [7]:
from calitp_data_analysis import calitp_color_palette as cp

In [8]:
pd.set_option("display.max_columns", None)

In [9]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"


In [10]:
display(HTML("<h1>Origin-Destination Big Data Analysis: SCAG POIs</h1>"))

In [11]:
display(HTML("This analysis uses Replica Data to identify the top trip start points within a Regional Transportation Planning Agency (RTPA) asnd determine what types of trips are occuring when and where."))

In [12]:
shape_data_name = "origins/replica-stm_origin_la-03_06_26-origin_layer.zip"

origins_name = "replica-stm_origin_la-03_06_26-trips_dataset.zip"

In [13]:
origins = _replica_utils.read_in_and_prep_replica_data_w_shp(origins_name, shape_data_name, file_type="")

In [14]:
# len(origins)

In [15]:
display(HTML("<h2><strong>Trip Counts</strong></h2>"))

In [16]:
display(HTML(f"The number of trips Traveling <strong>From</strong> the top 20 locations is <strong>{len(origins)}</strong>"))


In [17]:
assert origins.origin_bgrp_2020.nunique() == 20

In [18]:
origin_tracts_list = origins.origin_bgrp_2020.unique().tolist()

In [19]:
summary = _replica_utils.return_score_summary_single_df(origins, origin_tracts_list, "origin_bgrp_2020")

In [20]:
summary.columns = [col.replace('_', ' ').title() for col in summary.columns]


In [21]:
summary_melt =  pd.melt(
        summary,
        id_vars=["Trip Grouping"],
        value_vars=['Trip Grouping', 'Total Trips',
                    'N Auto Trips', 'Pct Auto Trips',
                    'N Tranist Trips', 'Pct Transit Trips',
                    'N Walking Trips', 'Pct Walking Trips',
                    'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
                    'Auto Mean Miles','Auto Median Miles', 'Auto Max Miles',
                    'Transit Mean Minutes','Transit Median Minutes', 'Transit Max Minutes',
                    'Transit Mean Miles','Transit Median Miles', 'Transit Max Miles',
                    'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes',
                    'Walking Mean Miles','Walking Median Miles', 'Walking Max Miles',
                   ],
        var_name="Metric",  # New column for original measurement names
        # value_name="_Value"
)

In [22]:
display(HTML("<h2><strong>Trip Type Summaries</strong></h2>"))

In [23]:
list_of_dfs = []


for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    modes_breakdown_subset = _replica_utils.get_mode_split(origins_subset, "origin_bgrp_2020")

    list_of_dfs.append(modes_breakdown_subset)

modes_breakdown = pd.concat(list_of_dfs, ignore_index=True)

In [24]:
modes_breakdown.columns = [col.replace('_', ' ').title() for col in modes_breakdown.columns]

In [25]:
modes_breakdown.sample()

,Trip Grouping,Mode,Pct Trips,Total Trips,N Blkgrs Dest
8,"1 (Tract 9202, Los Angeles, CA)",commercial,0.047733,360,343


In [26]:
alt.Chart(modes_breakdown).mark_bar().encode(
    x=alt.X("Trip Grouping:O", title = "Trip Origin"),
    y=alt.Y("Total Trips:Q", title="Total Number of Trips"),
    color=alt.Color("Mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    tooltip = ['Total Trips', 'Mode']
    ).properties(
    width=800,  
    height=300,
    title='Modes by Trip Type')

alt.Chart(...)

In [27]:
# alt.Chart(modes_breakdown).mark_bar().encode(
#     x=alt.X('Trip Grouping:O', title ="Trip Origin"),
#     y= alt.Y('Pct Trips:Q', title="Pct of Total Trips"),
#     color=alt.Color('Trip Grouping:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Grouping')),
#     column= alt.Column('Mode:N', title="Mode"),
#     tooltip=['Pct Trips', 'Mode', 'Trip Grouping']
# ).properties(width = 100, height = 400, title = "Modes by Trip Type")

In [28]:
display(HTML("<h3><strong>Closer Look at Auto, Transit & Walking Trips</strong></h3>"))

In [29]:
display(HTML("<strong>Tip:</strong> Use the dropdown menu to select different metrics to see on the chart."))

In [30]:
metrics_list = summary_melt["Metric"].unique().tolist()

metrics_dropdown = alt.binding_select(
        options=metrics_list,
        name="Metrics: ",
    )
    # Column that controls the bar charts
xcol_param = alt.selection_point(
        fields=["Metric"], value=metrics_list[0], bind=metrics_dropdown
    )

chart = (alt.Chart(summary_melt, title="Metric by Trip Types")
        .mark_bar()
        .encode(
            x=alt.X("value"),
            y=alt.Y("Trip Grouping"),
            color=alt.Color("Trip Grouping",
                                  scale=alt.Scale(
                                      range=cp.CALITP_CATEGORY_BRIGHT_COLORS))
        ).properties(width=400, height=350)
    ).add_params(xcol_param).transform_filter(xcol_param)

display(HTML("""
<style>
form.vega-bindings {
  position: absolute;
  left: 0px;
  top: -0px;
}
</style>
"""))

(chart)

alt.Chart(...)

In [31]:
display(HTML("<br>"
            "<br>"))

In [32]:
summary_long_all_miles = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Miles', 'Auto Median Miles', 'Auto Max Miles', 
        'Transit Mean Miles', 'Transit Median Miles', 'Transit Max Miles',
        'Walking Mean Miles', 'Walking Median Miles', 'Walking Max Miles'],
        var_name="Metric",  # New column for original measurement names
        value_name="Miles")

In [33]:
summary_long_all_min = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
        'Transit Mean Minutes', 'Transit Median Minutes', 'Transit Max Minutes',
        'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes'],
        var_name="Metric",  # New column for original measurement names
        value_name="Mintues")

In [34]:
display(HTML("<strong>Trip Length</strong>"))

In [35]:
alt.Chart(summary_long_all_min).mark_bar().encode(
    x="Mintues:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    row="Trip Grouping:O"
).properties(title="Travel Length by Trip Type & Mode")

alt.Chart(...)

In [36]:
alt.Chart(summary_long_all_miles).mark_bar().encode(
    x="Miles:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    row="Trip Grouping:O"
).properties(title="Travel Distance by Trip Type & Mode")

alt.Chart(...)

In [37]:
display(HTML("<h2><strong>Trip Timing</strong></h2>"))

In [38]:
display(HTML("<strong>Tip:</strong> You can zoom into each graph to better see the timing of the trips by mode"))

In [39]:
import datetime

In [40]:
# def add_hour_summary(analyses_study_data_list, station_geom_list): 
#     mode_col = "primary_mode"
    
#     dfs_list = []
#     dfs_time_bin_list = []
    
#     for analysis in analyses_study_data_list:
#         with get_fs().open(f"{gcs_path}replica_data_downloads/replica-socal_travel_analysis_{analysis}-trips_dataset.csv") as f:

#             df = to_snakecase(pd.read_csv(f, dtype={25: str, 26:str}, low_memory=False))


#             df['trip_start_time_test'] = df["trip_start_time"].apply(pd.to_datetime, errors="coerce")
#             df['trip_end_time_test'] = df["trip_end_time"].apply(pd.to_datetime, errors="coerce")
    


#             df_agg = (
#                 df.groupby(["primary_mode", "origin_tract_station_area"]).agg(
#                     trip_start_min=('trip_start_time_test', 'min'),
#                     trip_start_max=('trip_start_time_test', 'max'),
#                     trip_start_median=('trip_start_time_test', 'median'),
#                     trip_start_mean=('trip_start_time_test', 'mean'),
            
#                     trip_end_min=('trip_end_time_test', 'min'),
#                     trip_end_max=('trip_end_time_test', 'max'),
#                     trip_end_median=('trip_end_time_test', 'median'),
#                     trip_end_mean=('trip_end_time_test', 'mean'),
            
#                     trip_duration_minutes_min=('trip_duration_minutes', 'min'),
#                     trip_duration_minutes_max=('trip_duration_minutes', 'max'),
#                     trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#                     trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
            
#                     trip_duration_miles_min=('trip_distance_miles', 'min'),
#                     trip_duration_miles_max=('trip_distance_miles', 'max'),
#                     trip_duration_miles_median=('trip_distance_miles', 'median'),
#                     trip_duration_miles_mean=('trip_distance_miles', 'mean'),
#                     ).reset_index())

#             df_agg.columns = [col.replace('_', ' ').title() for col in df_agg.columns]
    
#             df_agg['Primary Mode'] = df_agg[f'Primary Mode'].apply(lambda x: x.replace('_', ' ').title())

            
#             dfs_list.append(df_agg)

#             df_time = df.set_index('trip_start_time_test')
#             result_time_bins = df_time.groupby(['primary_mode', 'origin_tract_station_area']).resample('30T').size().reset_index(name='trip_count')

#             dfs_time_bin_list.append(result_time_bins)
            

#     result_summary = pd.concat(dfs_list, ignore_index=True)
#     result_bin_summary = pd.concat(dfs_time_bin_list, ignore_index=True)
   
    
#     return result_summary, result_bin_summary
       

In [41]:
melt_dfs = []
time_duration_dfs =  []

for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    origins_subset_time_melt, origins_subset_time_melt_duration = _replica_utils.return_time_metrics(origins_subset, "trip_start_time", "trip_end_time")
    
    melt_dfs.append(origins_subset_time_melt)
    time_duration_dfs.append(origins_subset_time_melt_duration)



time_melt = pd.concat(melt_dfs, ignore_index=True)
time_duration_melt = pd.concat(time_duration_dfs, ignore_index=True)


In [42]:
alt.Chart(time_melt).mark_circle(size=150).encode(
    x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
    y='Metric:O',
    color=alt.Color("Primary Mode"),
    tooltip = ['Primary Mode', 'Metric', 'Time'],
).properties(height=400, width=600, title="Trips Start and End Times")

alt.Chart(...)

In [43]:
alt.Chart(time_duration_melt).mark_circle(size=150).encode(
    x=alt.X('Value:Q', title="Minutes"),
    y='Metric:O',
    color=alt.Color("Primary Mode"),
    tooltip = ['Primary Mode', 'Metric', 'Value']
).properties(height=400, width=600, title="Trip Duration for Trips")

alt.Chart(...)

In [44]:
n_trips_origin = (origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry"]).agg(
        {"activity_id": "nunique"}).reset_index()
n_trips_origin = n_trips_origin.set_geometry("geometry")


In [45]:
n_trips_dest = (origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry"]).agg(
        {"activity_id": "nunique"}).reset_index()
n_trips_dest = n_trips_dest.set_geometry("geometry")


In [46]:
display(HTML("<h2><strong>Trips by Census Block Groups</strong></h2>"))

In [47]:
n_trips_origin = n_trips_origin.rename(columns={"activity_id":"Number of Trips", "origin_bgrp_2020":"Origin Census BlockGroup"})

n_trips_dest = n_trips_dest.rename(columns={"destination_bgrp_2020":"Destination Census BlockGroup","activity_id":"Number of Trips"})

In [48]:
display(HTML("<h4>Trips by Origin</h4>"))

In [49]:
m = n_trips_origin.explore(name="Trips from Origins", column="Number of Trips", 
                cmap="YlOrRd")
display(m)

##### Testing Boxplot with all rows

In [50]:
# origin_tracts_list = ['1 (Tract 127, San Bernardino, CA)']

In [67]:
for tract in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == tract ]
    origins_subset = origins_subset[origins_subset["primary_mode"]!="other_travel_mode"]
    
    k = 1.5
    group_by_column = "primary_mode"
    value_column = "trip_duration_minutes"

    agg_stats = origins_subset.groupby(group_by_column)[value_column].describe()
    
    agg_stats["iqr"] = agg_stats["75%"] - agg_stats["25%"]
    agg_stats["min_"] = agg_stats["25%"] - k * agg_stats["iqr"]
    agg_stats["max_"] = agg_stats["75%"] + k * agg_stats["iqr"]
    data_points = origins_subset[[value_column, group_by_column]].merge(agg_stats.reset_index()[[group_by_column, "min_", "max_"]])
    
    # This will be the lower end of the whisker
    agg_stats["lower"] = (
        data_points[data_points[value_column] >= data_points["min_"]]
        .groupby(group_by_column)[value_column]
        .min()
    )
    # This will be the upper end of the whisker
    agg_stats["upper"] = (
        data_points[data_points[value_column] <= data_points["max_"]]
        .groupby(group_by_column)[value_column]
        .max()
    )
    
    agg_stats = agg_stats.reset_index()

    base = alt.Chart(agg_stats).encode(y = alt.Y("primary_mode:N", title="Primary Mode")).properties(title= f"Trip Durations for {tract}")

    rules = base.mark_rule().encode(
        x= alt.X("lower").title("Trip Duration (Minutes)"),
        x2="upper",)
    
    bars = base.mark_bar(size=14).encode(
        x="25%",
        x2="75%",
        color=alt.Color("primary_mode").legend(None),)
    
    ticks = base.mark_tick(color="white", size=14).encode(
        x="50%"
    )
    
    outliers = base.transform_flatten(
        flatten=["outliers"]
    ).mark_point(
        style="boxplot-outliers"
    ).encode(
        x="outliers:Q",
        color="primary_mode",
    )

    display(rules + bars + ticks)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

...

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

In [52]:
display(HTML("<h4>Trips by Destination</h3>"))


In [53]:
# m = n_trips_dest.explore(name="Trips by Destination", column="Number of Trips", cmap="YlOrRd")
# m

In [54]:
n_trips_dest_mode = (
    origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
        n_trips=("activity_id", "nunique"),
        
        trip_duration_minutes_median=('trip_duration_minutes', 'median'),
        trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
        
        trip_distance_miles_median=('trip_distance_miles', 'median'),
        trip_distance_miles_mean=('trip_distance_miles', 'mean'),
        
    #     trip_start_time_mean=('trip_start_time', 'mean'),
    #     trip_start_time_median=('trip_start_time', 'median'),
        
    #     trip_end_time_mean=('trip_end_time', 'mean'),
    #     trip_end_time_median=('trip_end_time', 'median'),
    
            ).reset_index()
    
n_trips_dest_mode = n_trips_dest_mode.set_geometry("geometry")

In [55]:
n_trips_origin_mode = (
    origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry", "primary_mode"]).agg(
        n_trips=("activity_id", "nunique"),
        
        trip_duration_minutes_median=('trip_duration_minutes', 'median'),
        trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
        
        trip_distance_miles_median=('trip_distance_miles', 'median'),
        trip_distance_miles_mean=('trip_distance_miles', 'mean'),
        
    #     trip_start_time_mean=('trip_start_time', 'mean'),
    #     trip_start_time_median=('trip_start_time', 'median'),
        
    #     trip_end_time_mean=('trip_end_time', 'mean'),
    #     trip_end_time_median=('trip_end_time', 'median'),
    
            ).reset_index()
        
n_trips_origin_mode = n_trips_origin_mode.set_geometry("geometry")

In [56]:
# alt.Chart(n_trips_dest_mode).mark_bar().encode(
#     x=alt.X('primary_mode', title ="Trip Mode"),
#     y= alt.Y('n_trips', title="Number of Trips"),
#     # color=alt.Color('destination_bgrp_2020:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Type')),
# ).properties(width = 500, height = 400, title = "Modes by Trip Type")

In [57]:
# n_trips_from_cp_mode.sample()

In [58]:
unique_modes = origins.primary_mode.unique()

In [59]:
unique_modes

array(['commercial', 'private_auto', 'auto_passenger',
       'other_travel_mode', 'public_transit', 'biking', 'walking',
       'on_demand_auto'], dtype=object)

In [60]:
# unique_modes = ['private_auto', 'public_transit', 'walking']

In [61]:
display(HTML("<h2>Trips Originating in the Top 20 tracts in the SCAG Area</h2>"))

In [62]:
# ##hashing out for now for saving
# replica_utils.return_mode_map(n_trips_to_cp_mode, routes, unique_modes, "to")

In [63]:
display(HTML("<h4>Trips Destinations of the Top 20 tracts in the SCAG area</h4>"))

In [64]:
###hashing out for now for saving
# replica_utils.return_mode_map(n_trips_from_cp_mode, routes, unique_modes, "from")

In [65]:
display(HTML("<h2>Raw Trip Data Sample</h2>"))

In [66]:
origins.sample(3)

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,tour_type,trip_start_local_hour,trip_end_local_hour,origin_zcta_2020,destination_zcta_2020,geometry
194740,3487889457211419297,"1 (Tract 2077.11, Los Angeles, CA)","2077.11 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 639.10, Orange, CA)","639.10 (Orange, CA)","Orange County, CA",California,on_demand_auto,shop,maintenance,20:59:00,21:46:29,47,37.0,unknown_vehicle_type,unknown_fuel_type,NaN,NaN,NaN,mixed_use,industrial,retail,retail,13227368571195032752,626434280139534382,29.0,female,hispanic_or_latino_origin,employed,in_person,45582.0,auto_passenger,2,94421.0,one,core,naics722515,multiple_units,not_attending_school,some_college,renter,spanish,"1 (Tract 992.26, Orange, CA)","992.26 (Orange, CA)","Orange County, CA",California,"3 (Tract 992.32, Orange, CA)","992.32 (Orange, CA)","Orange County, CA",California,work_based,20,21,90015,92626,"POLYGON ((-118.27266 34.04272, -118.27245 34.0..."
154581,2928473388294368475,"1 (Tract 9800.28, Los Angeles, CA)","9800.28 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 9203.13, Los Angeles, CA)","9203.13 (Los Angeles, CA)","Los Angeles County, CA",California,private_auto,eat,region_arrival,19:07:00,20:27:30,80,51.2,NaN,NaN,NaN,NaN,NaN,transportation_utilities,transportation_utilities,retail,retail,7603990648974417270,17997656721152286485,23.0,female,hispanic_or_latino_origin,employed,employed_not_working,47485.0,other_travel_mode,5,221282.0,three_plus,core,naics48_49,single_family,not_attending_school,high_school,owner,english,"2 (Tract 9203.22, Los Angeles, CA)","9203.22 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 9203.22, Los Angeles, CA)","9203.22 (Los Angeles, CA)","Los Angeles County, CA",California,undirected,19,20,90045,91321,"POLYGON ((-118.45233 33.94300, -118.43712 33.9..."
195769,9458769222245293558,"1 (Tract 9800.18, Los Angeles, CA)","9800.18 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 100.15, San Diego, CA)","100.15 (San Diego, CA)","San Diego County, CA",California,private_auto,home,region_arrival,13:56:00,16:17:47,141,126.4,NaN,NaN,NaN,NaN,NaN,transportation_utilities,transportation_utilities,single_family,single_family,1612842168666999456,7365420151358002797,76.0,female,hispanic_or_latino_origin,not_in_labor_force,unemployed_under_16_not_in_labor_force,8903.0,other_travel_mode,5,148748.0,three_plus,core,not_working,single_family,not_attending_school,k_12,owner,spanish,"2 (Tract 100.15, San Diego, CA)","100.15 (San Diego, CA)","San Diego County, CA",California,Does not have work/school location,Does not have work/school location,Does not have work/school location,Does not have work/school location,undirected,13,16,90806,92173,"POLYGON ((-118.18066 33.80511, -118.18066 33.8..."
